# Cafe Robot — Provided Baseline

Complete baseline code and pretrained weights are provided.
The baseline uses SW; the improved method must use SW, SE, NE, and NW. Both use the shared robot controller.

## Getting Started

1. Extract **student.zip**. For Colab, upload `student/cafe_project` to Drive and keep **dataset.zip** compressed.
2. Set **PROJECT_DIR** in section 0 to the folder containing `requirements.txt`. Review **DATA_DIR** and **SAVE_PATH** for local Jupyter.
3. **Run All** with the training call commented. This evaluates predictions, then runs picking with two CPU workers, using the supplied weights without retraining.
4. **Robot picking is slower:** for quick model checks, run only through section 9 and inspect the prediction figures/table. Continue with section 10 when ready; it reuses saved predictions.

Outputs are saved under **SAVE_PATH** (`runs/` in Drive for Colab).
Copy **baseline.ipynb** to **improved.ipynb**, set `MODEL_NAME = "improved"`, and implement your four-view method.
See **README.txt** for setup/submission and **DATASET_GUIDE.txt** for annotations. Follow this notebook for the updated evaluation order.


## 0. (Option) If you are using Colab
Extract the outer student.zip and upload its folder to Google Drive. Keep the inner dataset.zip compressed.
Set PROJECT_DIR below to the actual Drive code folder, run the cell, and approve Drive access.
Extract data to Colab's /content/dataset and save results/checkpoints to the code folder's runs/ in Drive.
Local Jupyter uses the paths in the else branch without mounting Drive.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    # Change this path if your code folder is elsewhere in Drive.
    PROJECT_DIR = Path("/content/drive/MyDrive/student/cafe_project")
    DATA_DIR = Path("/content/dataset")       # Local disk on the Colab server.
    SAVE_PATH = PROJECT_DIR / "runs"          # Keep results and weights in Drive.
else:
    PROJECT_DIR = Path('.').expanduser().resolve()
    DATA_DIR = Path("./dataset").expanduser().resolve()
    SAVE_PATH = Path("./runs").expanduser().resolve()

DATASET_ZIP = PROJECT_DIR / "dataset.zip"  # Set None if already extracted manually.
if not (PROJECT_DIR / "requirements.txt").is_file():
    raise FileNotFoundError("Set PROJECT_DIR to the code folder containing requirements.txt.")
print("Code:", PROJECT_DIR)
print("Dataset:", DATA_DIR)
print("Results:", SAVE_PATH)


## Install Packages
Use requirements.txt from the code folder configured above.
%pip installs packages into this notebook's Python environment. Run it again in a new runtime.


In [ ]:
requirements_path = str(PROJECT_DIR / "requirements.txt")
%pip install -r "{requirements_path}"


## 1. Prepare the Data
Extract DATASET_ZIP into DATA_DIR and save results under SAVE_PATH, as configured in section 0.
For manually extracted data, set DATASET_ZIP=None and select its DATA_DIR in section 0.
Completed extraction is reused. Prepare the data again after starting a new Colab runtime.


In [ ]:
from datetime import datetime
import os
import sys

SAVE_PATH.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(SAVE_PATH / "matplotlib_cache"))


In [ ]:
from common.data_setup import prepare_dataset

# Reuse the dataset if this ZIP has already been extracted.
DATA_DIR = prepare_dataset(DATASET_ZIP, DATA_DIR)


In [ ]:
import math
from pathlib import Path
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F

VIEWS = ("sw", "se", "ne", "nw")
COLORS = ("red", "blue", "green", "yellow", "black", "white")
PATTERNS = ("plain", "flower", "bird", "leaf", "stripes")


In [ ]:
MODEL_NAME = 'baseline'  # Set "improved" in your copied notebook.
MAX_LEVEL = 3          # Cumulative VAL scope: 1, 2, or 3. All TRAIN data remain available.
CHECKPOINT_PATH = (PROJECT_DIR / "weights/baseline.pth").resolve()
INPUT_VIEWS = ("sw",) if MODEL_NAME == "baseline" else VIEWS
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 17
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 0.0006
USE_WANDB = False      # Optional. False requires no W&B login.
WANDB_PROJECT = "cafe-cv-project"  # Used only when USE_WANDB=True.
WANDB_ENTITY = None
RUN_NAME = MODEL_NAME + "_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR = SAVE_PATH / RUN_NAME
print("Device:", DEVICE, "| Cameras:", INPUT_VIEWS)


## 2. Read the Data
L1: basic scenes where the target cup can be identified from SW.
L2: scenes containing cups with similar colors and patterns.
L3: scenes with occlusion and differences in what is visible across views.
TRAIN/VAL request lists are fixed for each level. Do not create new splits.
Training uses all 1,050 L1-L3 TRAIN requests. MAX_LEVEL selects only the cumulative VAL scope: 1 -> L1; 2 -> L1+L2; 3 -> L1+L2+L3.
VAL: L1 280 / L2 216 / L3 50 requests. TEST is private.


In [ ]:
import json
import random
import time
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)


In [ ]:
from common.request import parse_request


def read_json(path):
    with Path(path).open(encoding="utf-8") as stream:
        return json.load(stream)


def scene_folder(row, data_root):
    return Path(data_root) / row["scene_path"]


def read_requests(data_root, max_level, split):
    """Use all supplied TRAIN requests and cumulative VAL levels."""
    if split not in ("train", "val"):
        raise ValueError("Use TRAIN/VAL only for training and model selection.")
    if max_level not in (1, 2, 3):
        raise ValueError("Choose a challenge level from 1 to 3.")
    index = read_json(Path(data_root) / "index.json")
    if index["schema"] != "cafe_three_level_v10":
        raise ValueError("Select the current three-level dataset folder.")
    if index.get("revision") != "l1_l2_l3_only_v1":
        raise ValueError("Use the updated L1-L3 dataset and an empty extraction folder.")
    rows = []
    last_level = 3 if split == "train" else max_level
    for level in range(1, last_level + 1):
        metadata = read_json(Path(data_root) / f"L{level}.json")
        rows.extend(metadata["splits"][split])
    if len({row["sample_id"] for row in rows}) != len(rows):
        raise ValueError("Duplicate request IDs in the dataset.")
    for row in rows:
        # Normalize legacy dataset wording; cup colors, patterns, and targets stay the same.
        row["request"] = {**row["request"], **parse_request(row["request"]["text"])}
    return sorted(rows, key=lambda row: row["sample_id"])


In [ ]:
def load_observations(folder, views=VIEWS):
    """Read RGB, depth, and camera calibration for the specified views.

    Return observations[view]["rgb"] and observations[view]["geometry"].
    rgb: uint8 [H, W, 3], depth_m: float32 [H, W], depth_valid: bool [H, W].
    K: float64 [3, 3], camera_to_world: float64 [4, 4].
    Do not include ground truth, scene IDs, or levels in the returned observations.
    """
    scene_path = Path(folder)
    cameras = read_json(scene_path / "camera.json")
    observations = {}

    # Iterate over views because the number of requested views can vary.
    for view in views:
        rgb_path = scene_path / "rgb" / f"{view}.png"
        depth_path = scene_path / "depth" / f"{view}.tiff"
        valid_path = scene_path / "depth" / f"{view}_valid.png"

        with Image.open(rgb_path) as image:
            rgb_image = image.convert("RGB")
            rgb = np.array(rgb_image)

        # Read metric depth from TIFF; do not use depth_preview.png.
        with Image.open(depth_path) as image:
            depth_m = np.array(image, dtype=np.float32)

        with Image.open(valid_path) as image:
            valid_pixels = np.array(image)
        depth_valid = valid_pixels > 0

        camera = cameras[view]
        intrinsic_matrix = np.array(camera["K"], dtype=np.float64)
        camera_transform = np.array(camera["camera_to_world"], dtype=np.float64)

        geometry = {
            "depth_m": depth_m,
            "depth_valid": depth_valid,
            "K": intrinsic_matrix,
            "camera_to_world": camera_transform,
        }
        observations[view] = {"rgb": rgb, "geometry": geometry}

    return observations


def load_supervision(folder, request_id, view="sw"):
    """Read supervision for the requested cup. Never call this during inference.

    Return a copy of the selected view's entry in points.json.
    Load target_mask and target_parts as uint8 arrays instead of filenames.
    mask: target cup=255, other=0. parts: background=0, body=1, handle=2.
    When coordinate_loss_valid=False, reference_uv is None.
    """
    label_folder = Path(folder) / "label" / request_id
    labels_by_view = read_json(label_folder / "points.json")
    supervision = labels_by_view[view].copy()

    target_mask_path = label_folder / supervision["target_mask"]
    with Image.open(target_mask_path) as image:
        target_mask = np.array(image)

    target_parts_path = label_folder / supervision["target_parts"]
    with Image.open(target_parts_path) as image:
        target_parts = np.array(image)

    supervision["target_mask"] = target_mask
    supervision["target_parts"] = target_parts
    return supervision


In [ ]:
def image_to_tensor(rgb):
    """uint8 [H, W, 3] -> float32 [3, H, W], preserving the original RGB array."""
    image_tensor = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float()
    normalized_image = (image_tensor / 255.0 - 0.5) / 0.25
    return normalized_image


In [ ]:
class CupDataset(Dataset):
    def __init__(self, data_root, max_level=1, split="train", views=("sw",)):
        if split not in ("train", "val"):
            raise ValueError("Do not use TEST for training or checkpoint selection.")

        self.data_root = Path(data_root)
        # Use the request loader above without creating new splits.
        # Use all supplied TRAIN requests and the selected cumulative VAL levels.
        requests = read_requests(self.data_root, max_level, split)
        self.rows = [{**row, "view": view} for row in requests for view in views]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        """Build samples from one selected view, request IDs, and reference point labels.

        Return keys: image(float32 [3, 512, 512]), color(int), pattern(int),
        uv(float32 [2]), valid(bool). Coordinates are horizontal u, vertical v.
        Normalize valid points as (reference_uv + 0.5) / [W, H].
        Samples without point supervision use [NaN, NaN] and are excluded from location loss.
        """
        sample_info = self.rows[index]
        # Example: a row in L1.json -> dataset/scenes/scene001/
        folder = scene_folder(sample_info, self.data_root)
        request_id = sample_info["request_id"]

        view = sample_info["view"]
        observations = load_observations(folder, views=(view,))
        rgb = observations[view]["rgb"]
        label = load_supervision(folder, request_id, view=view)

        image_tensor = image_to_tensor(rgb)
        request = sample_info["request"]
        color_id = COLORS.index(request["color"])
        pattern_id = PATTERNS.index(request["pattern"])

        image_height, image_width = rgb.shape[:2]
        coordinate_valid = bool(label["coordinate_loss_valid"])
        if coordinate_valid:
            reference_uv = np.asarray(label["reference_uv"])
            image_size = np.array([image_width, image_height])
            normalized_uv = (reference_uv + 0.5) / image_size
        else:
            normalized_uv = np.full(2, np.nan)

        target_uv = torch.tensor(normalized_uv, dtype=torch.float32)
        sample = {
            "image": image_tensor,
            "color": color_id,
            "pattern": pattern_id,
            "uv": target_uv,
            "valid": coordinate_valid,
        }
        return sample


## 3. Baseline Model and Loss
A six-layer CNN takes RGB and the requested color/pattern, then predicts a location heatmap and observability logit.
Observability is not grasp success probability. All layers and the forward pass are provided.


In [ ]:
def encode_request(color_id, pattern_id):
    """Convert integer color and pattern IDs into a single request vector.

    Input: color_id [B], pattern_id [B], where B is the batch size.
    Output: one-hot vector [B, 11] with 6 color and 5 pattern entries.
    For example, color ID 0 becomes [1, 0, 0, 0, 0, 0].
    """
    color_features = F.one_hot(color_id, num_classes=len(COLORS))
    pattern_features = F.one_hot(pattern_id, num_classes=len(PATTERNS))

    # Concatenate each sample's color and pattern vectors along dim=1.
    request_features = torch.cat([color_features, pattern_features], dim=1)
    return request_features


class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Concatenate 3 RGB channels and 11 request channels: 14 input channels.
        input_channels = 3 + len(COLORS) + len(PATTERNS)

        # Define each layer explicitly to show channel and resolution changes.
        # Preserve encoder layer order for compatibility with supplied checkpoints.
        self.encoder = nn.Sequential(
            # Block 1: [B, 14, 512, 512] -> [B, 16, 256, 256]
            nn.Conv2d(
                input_channels, 16, kernel_size=3,
                stride=2, padding=1, dilation=1, bias=False,
            ),
            nn.GroupNorm(num_groups=4, num_channels=16),
            nn.ReLU(),

            # Block 2: [B, 16, 256, 256] -> [B, 32, 128, 128]
            nn.Conv2d(
                16, 32, kernel_size=3,
                stride=2, padding=1, dilation=1, bias=False,
            ),
            nn.GroupNorm(num_groups=4, num_channels=32),
            nn.ReLU(),

            # Block 3: preserve spatial resolution and increase feature channels to 64.
            nn.Conv2d(
                32, 64, kernel_size=3,
                stride=1, padding=1, dilation=1, bias=False,
            ),
            nn.GroupNorm(num_groups=4, num_channels=64),
            nn.ReLU(),

            # Block 4: dilation=2 expands the receptive field.
            nn.Conv2d(
                64, 64, kernel_size=3,
                stride=1, padding=2, dilation=2, bias=False,
            ),
            nn.GroupNorm(num_groups=4, num_channels=64),
            nn.ReLU(),

            # Block 5: dilation=4. Output shape: [B, 64, 128, 128].
            nn.Conv2d(
                64, 64, kernel_size=3,
                stride=1, padding=4, dilation=4, bias=False,
            ),
            nn.GroupNorm(num_groups=4, num_channels=64),
            nn.ReLU(),

            # Block 6: dilation=8. No further spatial downsampling.
            nn.Conv2d(
                64, 64, kernel_size=3,
                stride=1, padding=8, dilation=8, bias=False,
            ),
            nn.GroupNorm(num_groups=4, num_channels=64),
            nn.ReLU(),
        )

        # The location head produces one logit per spatial position.
        self.location = nn.Conv2d(64, 1, kernel_size=1)

        # The observability head produces one logit from global image features.
        self.visibility = nn.Linear(64, 1)

    def forward(self, rgb, color_id, pattern_id):
        """Encode the request, combine it with RGB, extract features, and run both heads.

        RGB input: [B, 3, H, W]. Default images have H=W=512.
        For 512-pixel input: heatmap_logits [B, 128, 128], observable_logit [B].
        Both outputs are logits, not probabilities.
        """
        image_height = rgb.shape[2]
        image_width = rgb.shape[3]

        # 1. Encode the request and cast integer one-hot vectors to the RGB dtype.
        request_features = encode_request(color_id, pattern_id).to(dtype=rgb.dtype)

        # 2. Provide the same request vector at every pixel.
        # [B, 11] -> [B, 11, 1, 1] -> [B, 11, H, W]
        request_map = request_features[:, :, None, None].expand(-1, -1, image_height, image_width)

        # 3. Concatenate channels, then extract features with the CNN.
        combined_input = torch.cat([rgb, request_map], dim=1)
        image_features = self.encoder(combined_input)

        # 4. Location head: [B, 64, H/4, W/4] -> [B, 1, H/4, W/4].
        heatmap_logits = self.location(image_features).squeeze(1)

        # 5. Observability head: spatial mean [B, 64] -> logit [B, 1] -> [B].
        pooled_features = image_features.mean(dim=(2, 3))
        observable_logit = self.visibility(pooled_features).squeeze(1)

        predictions = {
            "heatmap_logits": heatmap_logits,
            "observable_logit": observable_logit,
        }
        return predictions


Apply location loss only to valid reference points and observability loss to every sample.


In [ ]:
def compute_loss(output, uv, coordinate_valid, positive_fraction):
    """Compute location loss + 0.25 * observability loss.

    output: heatmap_logits [B, H, W] and observable_logit [B] from the model.
    uv: normalized reference coordinates [B, 2]; unavailable supervision uses NaN.
    coordinate_valid: Boolean mask [B], True when point supervision is available.
    positive_fraction: fraction of TRAIN samples with valid point supervision.
    Return a scalar Tensor that supports backpropagation.
    """
    heatmap_logits = output["heatmap_logits"]
    observable_logits = output["observable_logit"]
    valid_samples = coordinate_valid.bool()

    # 1. Apply location loss only where coordinate_loss_valid is True.
    # Select valid rows before computing with potentially NaN coordinates.
    if valid_samples.any():
        heatmap_height, heatmap_width = heatmap_logits.shape[-2:]
        valid_uv = uv[valid_samples]

        y_coordinates = torch.arange(heatmap_height, device=heatmap_logits.device)
        x_coordinates = torch.arange(heatmap_width, device=heatmap_logits.device)
        grid_y, grid_x = torch.meshgrid(
            y_coordinates, x_coordinates, indexing="ij"
        )

        # Map normalized coordinates to heatmap pixel-center coordinates.
        heatmap_size = uv.new_tensor([heatmap_width, heatmap_height])
        target_centres = valid_uv * heatmap_size - 0.5
        centre_x = target_centres[:, 0, None, None]
        centre_y = target_centres[:, 1, None, None]

        squared_distance_x = (grid_x[None] - centre_x) ** 2
        squared_distance_y = (grid_y[None] - centre_y) ** 2
        squared_distance = squared_distance_x + squared_distance_y

        # Supervise with a Gaussian around the reference point; sigma is in heatmap pixels.
        sigma = 1.2
        target_heatmap = torch.exp(-squared_distance / (2 * sigma ** 2))
        target_sum = target_heatmap.sum(dim=(1, 2), keepdim=True)
        target_heatmap = target_heatmap / target_sum

        # [valid samples, H, W] -> [valid samples, H*W].
        # Cross entropy treats all spatial positions as candidate locations.
        valid_logits = heatmap_logits[valid_samples]
        flat_logits = valid_logits.flatten(start_dim=1)
        log_probabilities = flat_logits.log_softmax(dim=1)
        flat_targets = target_heatmap.flatten(start_dim=1)

        loss_per_location = flat_targets * log_probabilities
        location_loss_per_sample = -loss_per_location.sum(dim=1)
        location_loss = location_loss_per_sample.mean()
    else:
        # Keep the computation graph connected even when no points have supervision.
        location_loss = heatmap_logits.sum() * 0.0

    # 2. Supervise observability with coordinate_loss_valid for every sample.
    # Use TRAIN class frequencies so the rarer label is not ignored.
    visible_weight = 0.5 / positive_fraction
    hidden_weight = 0.5 / (1 - positive_fraction)
    sample_weights = torch.where(
        valid_samples, visible_weight, hidden_weight
    )
    observable_targets = valid_samples.float()
    observable_loss_per_sample = F.binary_cross_entropy_with_logits(
        observable_logits,
        observable_targets,
        reduction="none",
    )
    weighted_observable_losses = observable_loss_per_sample * sample_weights
    observable_loss = weighted_observable_losses.mean()

    total_loss = location_loss + 0.25 * observable_loss
    return total_loss


## 4. Training Code — Run Only When Needed
Read train_epoch, validate_epoch, and train_model in order.
validate_epoch computes loss for checkpoint selection; the final section evaluates actual robot picking.
Executing these function definitions does not start training.
Keep the training call commented for baseline analysis. Modify and run it only when your improved method needs training.
Choose freely among the available training inputs, including RGB, depth, and masks. Ground truth is excluded from inference.


In [ ]:
def train_epoch(model, loader, optimizer, device, positive_fraction):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for batch in tqdm(loader, desc="TRAIN", leave=False):
        images = batch["image"].to(device)
        colors = batch["color"].to(device)
        patterns = batch["pattern"].to(device)
        target_uv = batch["uv"].to(device)
        coordinate_valid = batch["valid"].to(device)

        optimizer.zero_grad(set_to_none=True)
        output = model(images, colors, patterns)
        loss = compute_loss(output, target_uv, coordinate_valid, positive_fraction)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        batch_size = images.shape[0]
        total_loss = total_loss + loss.detach().item() * batch_size
        total_samples = total_samples + batch_size
    average_loss = total_loss / total_samples
    return average_loss


In [ ]:
def validate_epoch(model, loader, device, positive_fraction):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for batch in tqdm(loader, desc="VAL", leave=False):
            images = batch["image"].to(device)
            colors = batch["color"].to(device)
            patterns = batch["pattern"].to(device)
            target_uv = batch["uv"].to(device)
            coordinate_valid = batch["valid"].to(device)

            output = model(images, colors, patterns)
            loss = compute_loss(output, target_uv, coordinate_valid, positive_fraction)
            batch_size = images.shape[0]
            total_loss = total_loss + loss.item() * batch_size
            total_samples = total_samples + batch_size
    average_loss = total_loss / total_samples
    return average_loss


In [ ]:
def train_model():
    train_dataset = CupDataset(data_root=DATA_DIR, max_level=MAX_LEVEL, split="train", views=INPUT_VIEWS)
    val_dataset = CupDataset(data_root=DATA_DIR, max_level=MAX_LEVEL, split="val", views=INPUT_VIEWS)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = BaselineCNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.0001)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=min(0.00001, LEARNING_RATE))

    # Compute observability class weights from TRAIN labels only.
    visible_labels = []
    for row in train_dataset.rows:
        folder = scene_folder(row, DATA_DIR)
        labels = read_json(folder / "label" / row["request_id"] / "points.json")
        view = row.get("view", "sw")
        visible_labels.append(labels[view]["coordinate_loss_valid"])
    positive_fraction = float(np.clip(np.mean(visible_labels), 0.01, 0.99))

    training_dir = SAVE_PATH / (MODEL_NAME + "_training_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    training_dir.mkdir(parents=True, exist_ok=False)
    checkpoint_path = training_dir / "best.pth"
    config = {
        "use_wandb": USE_WANDB, "model_name": MODEL_NAME, "max_level": MAX_LEVEL, "epochs": EPOCHS,
        "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "seed": SEED,
        "train_samples": len(train_dataset), "val_samples": len(val_dataset),
        "device": DEVICE, "gpu": torch.cuda.get_device_name() if DEVICE == "cuda" else None,
    }
    with (training_dir / "config.json").open("w", encoding="utf-8") as stream:
        json.dump(config, stream, ensure_ascii=False, indent=2)

    run = None
    if USE_WANDB:
        import wandb
        run = wandb.init(
            project=WANDB_PROJECT, entity=WANDB_ENTITY, name=training_dir.name,
            config=config, mode=os.environ.get("WANDB_MODE", "online"), dir=str(training_dir), save_code=False,
            settings=wandb.Settings(disable_git=True))
    best_val_loss = float("inf")
    history = []
    started = time.perf_counter()
    failed = True
    try:
        for epoch in range(1, EPOCHS + 1):
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            epoch_start = time.perf_counter()

            train_loss = train_epoch(model, train_loader, optimizer, DEVICE, positive_fraction)
            val_loss = validate_epoch(model, val_loader, DEVICE, positive_fraction)
            scheduler.step()
            if not np.isfinite([train_loss, val_loss]).all():
                raise ValueError("Training or validation loss is not finite.")

            if DEVICE == "cuda":
                torch.cuda.synchronize()
            record = {
                "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                "epoch_seconds": time.perf_counter() - epoch_start,
            }
            history.append(record)
            if run is not None:
                run.log(record, step=epoch)
            print(record)

            # Save directly under SAVE_PATH without a separate backup step.
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                checkpoint = {
                    "state_dict": model.state_dict(), "epoch": epoch, "max_level": MAX_LEVEL,
                    "format": "cafe-student-baseline-v3", "seed": SEED,
                }
                torch.save(checkpoint, checkpoint_path)
            with (training_dir / "history.json").open("w", encoding="utf-8") as stream:
                json.dump(history, stream, ensure_ascii=False, indent=2)
            pd.DataFrame(history).to_csv(training_dir / "history.csv", index=False)

        if run is not None:
            run.summary["best_val_loss"] = best_val_loss
            run.summary["training_seconds"] = time.perf_counter() - started
        failed = False
    finally:
        if run is not None:
            run.finish(exit_code=1 if failed else 0)

    print("Checkpoint:", checkpoint_path)

    return checkpoint_path


### Optional W&B Logging and Training
W&B is optional. Leave USE_WANDB=False for training and evaluation without login.
To enable it, set USE_WANDB=True in the configuration cell and rerun this login cell.
WANDB_MODE=offline supports optional local W&B logs without authentication.
Never put API keys in code or submissions. Local CSV/JSON results are saved even when W&B is off.
Uncommenting the training call saves the best weights in a new folder and connects them to later prediction/evaluation.
Supplied weights are not overwritten. After training, comment out the call again and
set CHECKPOINT_PATH to the saved weights when evaluating them later.


In [ ]:
if USE_WANDB:
    import wandb
    if os.environ.get("WANDB_MODE", "online") != "offline":
        wandb.login()
else:
    print("W&B is off. Training and evaluation results will be saved as local files.")


In [ ]:
# Baseline analysis: leave this line commented to use the provided weights.
# Uncomment only to train your improved model. Predictor will load the returned weights.
# CHECKPOINT_PATH = train_model()


## 5. Coordinate Transforms
Use depth and K to obtain camera coordinates, then camera_to_world to obtain world coordinates.
Return coordinates in meters. The robot uses this position as the reference for execution.


In [ ]:
def pixel_to_camera(uv, depth_m, K):
    """Convert pixels and camera-z depth to camera coordinates.

    Input: uv (..., 2), depth_m (...), K (3, 3). Output: (..., 3).
    Support both a single point [u, v] and a batch of points [N, 2].
    Formula: camera_point = depth * inverse(K) * [u, v, 1].
    """
    pixel_uv = np.asarray(uv, dtype=float)
    depth_values = np.asarray(depth_m, dtype=float)
    intrinsic_matrix = np.asarray(K, dtype=float)

    if pixel_uv.shape[-1:] != (2,):
        raise ValueError("uv must end with two coordinates: [u, v]")
    if depth_values.shape != pixel_uv.shape[:-1]:
        raise ValueError("Each pixel must have one depth value")
    if intrinsic_matrix.shape != (3, 3):
        raise ValueError("K must be a 3-by-3 matrix")
    if not np.isfinite(pixel_uv).all():
        raise ValueError("Pixel coordinates must be finite")
    if not np.isfinite(depth_values).all() or (depth_values <= 0).any():
        raise ValueError("Depth must be finite and positive")

    # Append 1 to [u, v] to obtain homogeneous pixel coordinates.
    ones_shape = pixel_uv.shape[:-1] + (1,)
    ones = np.ones(ones_shape)
    homogeneous_pixels = np.concatenate([pixel_uv, ones], axis=-1)

    # Points are row vectors, so multiply by the transposed rotation on the right.
    inverse_intrinsics = np.linalg.inv(intrinsic_matrix)
    camera_rays = homogeneous_pixels @ inverse_intrinsics.T
    camera_points = camera_rays * depth_values[..., None]
    return camera_points


def camera_to_world(points_camera, transform):
    """Apply rotation and translation to camera coordinates.

    Input: points_camera (..., 3), transform (4, 4). Output: (..., 3).
    transform is camera_to_world; do not invert it again.
    """
    camera_points = np.asarray(points_camera, dtype=float)
    camera_transform = np.asarray(transform, dtype=float)

    if camera_points.shape[-1:] != (3,):
        raise ValueError("Camera points must end with three coordinates")
    if camera_transform.shape != (4, 4):
        raise ValueError("camera_to_world must be a 4-by-4 matrix")
    if not np.isfinite(camera_points).all():
        raise ValueError("Camera points must be finite")
    if not np.isfinite(camera_transform).all():
        raise ValueError("camera_to_world must be finite")

    rotation = camera_transform[:3, :3]
    translation = camera_transform[:3, 3]
    rotated_points = camera_points @ rotation.T
    world_points = rotated_points + translation
    return world_points


def pixel_to_world(uv, depth_m, K, transform):
    """Apply both coordinate transforms in order. Return coordinates in meters."""
    camera_points = pixel_to_camera(uv, depth_m, K)
    world_points = camera_to_world(camera_points, transform)
    return world_points


def point_from_depth(pixel_uv, geometry):
    """Read depth at the predicted pixel and apply the coordinate transforms.

    Use the nearest integer pixel without interpolating across object boundaries.
    Return world coordinates [x, y, z] and the sampled integer pixel [u, v].
    """
    predicted_uv = np.asarray(pixel_uv, dtype=float)
    depth_map = geometry["depth_m"]

    if predicted_uv.shape != (2,) or not np.isfinite(predicted_uv).all():
        raise ValueError("pixel_uv must contain two finite values")

    image_height, image_width = depth_map.shape
    pixel_u, pixel_v = predicted_uv
    inside_width = 0 <= pixel_u <= image_width - 1
    inside_height = 0 <= pixel_v <= image_height - 1
    if not inside_width or not inside_height:
        raise ValueError("Predicted pixel is outside the image")

    # Array indexing is [row, column], so read depth at [v, u].
    rounded_uv = np.floor(predicted_uv + 0.5).astype(int)
    sample_u, sample_v = rounded_uv
    depth_m = float(depth_map[sample_v, sample_u])
    depth_valid = geometry["depth_valid"][sample_v, sample_u]
    if not depth_valid or not np.isfinite(depth_m) or depth_m <= 0:
        raise ValueError("Invalid depth at the predicted pixel")

    sampled_pixel = np.array([sample_u, sample_v])
    world_point = pixel_to_world(
        sampled_pixel,
        depth_m,
        geometry["K"],
        geometry["camera_to_world"],
    )
    world_point = np.asarray(world_point, dtype=float)
    used_pixel = [int(sample_u), int(sample_v)]
    return world_point, used_pixel


## 6. Prediction Interface
Keep Predictor(checkpoint=None, device="cpu") and predict(observations, request).
Inputs contain RGB, depth, camera matrices, and the request. Ground-truth masks and reference points are excluded.
PICK returns selected_view and world_xyz_m. pixel_uv is optional.
NO_PICK returns reason. Rejected requests remain in the success-rate denominator.
Inference cells have the cafe.inference tag. Add this tag to any new inference cells.


In [ ]:
def decode_heatmap(logits):
    """Convert heatmap logits [B, H, W] to normalized coordinates [B, 2].

    Compute softmax and a weighted mean only within a 5x5 window around the peak.
    A full-image weighted average could place the point between different cups.
    Return (grid coordinates + 0.5) / [W, H].
    """
    batch_size, heatmap_height, heatmap_width = logits.shape

    # 1. Find the maximum-scoring position in each heatmap.
    flat_logits = logits.flatten(start_dim=1)
    peak_indices = flat_logits.argmax(dim=1)
    peak_x = peak_indices % heatmap_width
    peak_y = peak_indices // heatmap_width

    y_coordinates = torch.arange(heatmap_height, device=logits.device)
    x_coordinates = torch.arange(heatmap_width, device=logits.device)
    grid_y, grid_x = torch.meshgrid(
        y_coordinates, x_coordinates, indexing="ij"
    )

    # 2. Keep positions within two pixels of the peak along each axis.
    # Reshape [B] to [B, 1, 1] to compare each peak against the entire grid.
    peak_x = peak_x[:, None, None]
    peak_y = peak_y[:, None, None]
    distance_x = (grid_x[None] - peak_x).abs()
    distance_y = (grid_y[None] - peak_y).abs()
    local_mask = (distance_x <= 2) & (distance_y <= 2)

    # 3. Suppress scores outside the window so their softmax probability is near zero.
    masked_logits = logits.masked_fill(~local_mask, -1e9)
    flat_masked_logits = masked_logits.flatten(start_dim=1)
    flat_probabilities = flat_masked_logits.softmax(dim=1)
    probabilities = flat_probabilities.reshape(
        batch_size, heatmap_height, heatmap_width
    )

    # 4. Use probabilities as weights to obtain subpixel coordinates.
    weighted_x = probabilities * grid_x
    weighted_y = probabilities * grid_y
    expected_x = weighted_x.sum(dim=(1, 2))
    expected_y = weighted_y.sum(dim=(1, 2))

    centre_x = expected_x + 0.5
    centre_y = expected_y + 0.5
    centre_uv = torch.stack([centre_x, centre_y], dim=1)
    heatmap_size = logits.new_tensor([heatmap_width, heatmap_height])
    normalized_uv = centre_uv / heatmap_size
    return normalized_uv


class Predictor:
    def __init__(self, checkpoint=None, device="cpu"):
        if checkpoint is None:
            raise ValueError("The baseline requires a checkpoint path.")

        self.device = device
        self.model = BaselineCNN()
        self.model = self.model.to(device)
        self.model.eval()

        saved_checkpoint = torch.load(
            checkpoint, map_location="cpu", weights_only=True
        )
        self.model.load_state_dict(saved_checkpoint["state_dict"])

    @torch.inference_mode()
    def predict(self, observations, request):
        """Connect the SW image, model prediction, pixel depth, and world coordinates.

        observations contains rgb and geometry for each view.
        request contains text, color, and pattern.
        For PICK, return selected_view, pixel_uv, world_xyz_m, and
        reference_observable_probability.
        Return NO_PICK and a reason if depth is invalid.
        """
        if set(observations) != {"sw"}:
            raise ValueError("The baseline uses SW only. Implement four-view processing in the improved model.")

        # 1. The baseline uses one SW image without modifying the input arrays.
        selected_view = "sw"
        observation = observations[selected_view]
        rgb = observation["rgb"]

        image_tensor = image_to_tensor(rgb)
        image_batch = image_tensor.unsqueeze(0)
        image_batch = image_batch.to(self.device)

        color_id = COLORS.index(request["color"])
        pattern_id = PATTERNS.index(request["pattern"])
        color_batch = torch.tensor([color_id], device=self.device)
        pattern_batch = torch.tensor([pattern_id], device=self.device)

        # 2. Convert the predicted heatmap position to original image coordinates.
        predictions = self.model(image_batch, color_batch, pattern_batch)
        normalized_uv_batch = decode_heatmap(predictions["heatmap_logits"])
        normalized_uv = normalized_uv_batch[0]
        normalized_uv = normalized_uv.cpu().numpy()

        image_height, image_width = rgb.shape[:2]
        image_size = np.array([image_width, image_height])
        pixel_uv = normalized_uv * image_size - 0.5

        observable_probabilities = predictions["observable_logit"].sigmoid()
        observable_probability = float(observable_probabilities[0])
        result = {
            "selected_view": selected_view,
            "pixel_uv": pixel_uv.tolist(),
            "reference_observable_probability": observable_probability,
        }

        # 3. Use depth and camera calibration to obtain world coordinates.
        # Observability is not grasp success probability; a low score alone does not reject a pick.
        try:
            world_point, used_pixel = point_from_depth(
                pixel_uv, observation["geometry"]
            )
        except ValueError as error:
            result["status"] = "NO_PICK"
            result["reason"] = str(error)
            return result

        # The shared controller uses this coordinate as the grasp target reference.
        result["status"] = "PICK"
        result["world_xyz_m"] = world_point.tolist()
        return result


## 7. Load Weights and Predict a VAL Request
Load CHECKPOINT_PATH. The baseline uses the supplied weights by default.
If you ran training above, the improved model loads the newly saved best weights.
Reference-point accuracy and actual picking success are different measurements.


In [ ]:
from common.interface import check_output_format

val_rows = read_requests(DATA_DIR, MAX_LEVEL, "val")
predictor = Predictor(CHECKPOINT_PATH, device=DEVICE)
sample = val_rows[0]
observations = load_observations(scene_folder(sample, DATA_DIR), views=INPUT_VIEWS)
request = {key: sample["request"][key] for key in ("text", "color", "pattern")}
prediction = predictor.predict(observations, request)
check_output_format(prediction, observations)
print(sample["sample_id"], request["text"])
print(prediction)


## 8. Prediction Visualization and Optional Robot Demo
Inspect the predicted image point first. Section 9 evaluates predictions without robot execution.
Use the optional demo below for a selected case, or section 10 for full picking evaluation.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

selected_view = prediction.get("selected_view", INPUT_VIEWS[0])
figure, axis = plt.subplots(figsize=(6, 6))
axis.imshow(observations[selected_view]["rgb"])
pixel = prediction.get("pixel_uv")
if pixel is not None:
    axis.plot(pixel[0], pixel[1], "mx", markersize=12, markeredgewidth=2)
axis.set_title(f"{request['color']} / {request['pattern']} | {selected_view.upper()}")
axis.axis("off")
RUN_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(RUN_DIR / "prediction.png", bbox_inches="tight")
plt.close(figure)
display(DisplayImage(filename=str(RUN_DIR / "prediction.png")))


### Optional: Robot Picking GIF for One Request
Set DEMO_SAMPLE_ID to a VAL request ID to execute the robot and save the result.
Leave None to skip the demo. Results appear inline without opening a separate window.
The cyan target outline is for explanation only and is never passed to the model.


In [ ]:
from common.visualization import visualize_picking

DEMO_SAMPLE_ID = None  # Example: "scene001/request01"
if DEMO_SAMPLE_ID is not None:
    demo_output = RUN_DIR / "demos" / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    demo = visualize_picking(
        predictor, DEMO_SAMPLE_ID, max_level=MAX_LEVEL, data_root=DATA_DIR,
        model_name=MODEL_NAME, views=INPUT_VIEWS, output=demo_output)


## 9. Prediction Evaluation — Run This First
**Robot picking is slower.** For quick model checks, stop after this section; Run All continues to picking in section 10.
The common evaluator saves predictions and diagnostic results without executing the robot.

- **Point on target:** predicted pixels on the requested cup's visible mask / all requests. This does not imply a valid grasp.
- **Reference error (pixels):** mean distance to labeled points in the selected view; reference_pairs is the number compared. L3 has no valid 2D reference points, so its error is N/A.
- These are diagnostic indicators, not picking scores. NO_PICK/errors count as mask misses; missing references are excluded from distance averages. Occluded 3D points may miss the visible mask.

Labels are used only after prediction. Per-request details are saved in prediction_details.csv; predictions and robot commands are saved in predictions.json.

Estimated runtime on Colab: approximately 15 minutes.

In [ ]:
from common.validation import evaluate_predictions

PREDICTION_DIR, prediction_table = evaluate_predictions(
    predictor, max_level=MAX_LEVEL, data_root=DATA_DIR,
    views=INPUT_VIEWS, model_name=MODEL_NAME, output_root=RUN_DIR,
    checkpoint=CHECKPOINT_PATH, use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY,
)
display(prediction_table)
print("Saved predictions:", PREDICTION_DIR)


## 10. Picking Evaluation — Reuse Saved Predictions
**Run All includes this slower step.** It reuses saved predictions and runs two CPU workers without calling the model again.
To reuse an earlier run, set PREDICTION_DIR to its saved folder. After a restart, rerun setup and skip prediction/model loading if only evaluating saved predictions.

Success means lifting and stably holding the requested cup; placement is not evaluated.
All saved requests remain in the denominator, including rejections, wrong-cup picks, and model errors. GIFs are not generated here.
Results are saved in a new picking_... folder inside PREDICTION_DIR. This step measures actual picking success.


In [ ]:
from common.validation import evaluate_saved_picking

picking_table = evaluate_saved_picking(
    PREDICTION_DIR, data_root=DATA_DIR, robot_workers=2,
    use_wandb=USE_WANDB,
    wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY,
)
display(picking_table)
